In [3]:
#----------------------------------------------------------#
#         Program: Predict Dogecoin 2021/02/10             #
#               All rights reserved 2021                   #
#----------------------------------------------------------#
#     From: Ekobots Innovation Ltda - www.ekobots.com      #
#       by: Juan Sirgado y Antico - www.jsya.com.br        #
#----------------------------------------------------------#
# Description:                                             # 
# Predicts the price of Dogecoin for the next 30 days      #
#----------------------------------------------------------#
import numpy as np 
import pandas as pd
import postgresql as ps

In [4]:
#Connecta com o banco de dados db_bovespa
try:
    db = ps.open(host = "127.0.0.1",
                         port = "5432",
                         database = "db_cryptocoin",
                         user = "postgres",
                         password = "sirgadoa")
    print ("ok-open")
except ps.exceptions.UniqueError:
    print ("erro-open")

ok-open


In [5]:
#=================================================================================================================#
# https://finance.yahoo.com/quote/DOGE-USD/history
# Download to Doge-USD.csv
#=================================================================================================================#

In [6]:
#Load the data
from google.colab import files # Use to load data on Google Colab
uploaded = files.upload() # Use to load data on Google Colab

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
#Store the data into the variable df
df = pd.read_csv('Doge-USD.csv')
df.head(7)

In [ ]:
#Remove the Date column
df.drop(['Date'], 1, inplace=True)

In [ ]:
#Show the first 7 rows of the new data set
df.head(12)

In [ ]:
#A variable for predicting 'n' days out into the future
prediction_days = 30 #n = 30 days

#Create another column (the target or dependent variable) shifted 'n' units up
df['Prediction'] = df[['Price']].shift(-prediction_days)

In [ ]:
#Show the first 7 rows of the new data set
df.head(12)

In [ ]:
#Show the last 7 rows of the new data set
df.tail(12)

In [ ]:
#CREATE THE INDEPENDENT DATA SET (X)

# Convert the dataframe to a numpy array and drop the prediction column
X = np.array(df.drop(['Prediction'],1))

#Remove the last 'n' rows where 'n' is the prediction_days
X= X[:len(df)-prediction_days]
print(X)

In [ ]:
#CREATE THE DEPENDENT DATA SET (y) 

# Convert the dataframe to a numpy array (All of the values including the NaN's) 
y = np.array(df['Prediction'])  

# Get all of the y values except the last 'n' rows 
y = y[:-prediction_days] 
print(y)

In [ ]:
# Split the data into 80% training and 20% testing
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Set prediction_days_array equal to the last 30 rows of the original data set from the price column
prediction_days_array = np.array(df.drop(['Prediction'],1))
[-prediction_days:]
print(prediction_days_array)

In [ ]:
from sklearn.svm import SVR

# Create and train the Support Vector Machine
# Create the Support Vector Regression model using the radial basis function (rbf), and train the model.

#Create the model
svr_rbf = SVR(kernel='rbf', C=1e3, gamma=0.00001)

#Train the model
svr_rbf.fit(x_train, y_train)

In [ ]:
# Testing Model: Score returns the accuracy of the prediction. 
# The best possible score is 1.0
svr_rbf_confidence = svr_rbf.score(x_test, y_test)

#Show the accuracy of the model on the testing data sets
print("svr_rbf accuracy: ", svr_rbf_confidence)

In [ ]:
# Print the predicted value
svm_prediction = svr_rbf.predict(x_test)
print("Prediction values: ")
print(svm_prediction)

print()

#Print the actual values
print("Test real values: ")
print(y_test)

In [ ]:
# Print the model predictions for the next 'n=30' days
svm_prediction = svr_rbf.predict(prediction_days_array)
print(svm_prediction)

In [ ]:
#Print the actual price for the next 'n' days, n=prediction_days=30 
df.tail(prediction_days)

In [4]:
#Delete os registros das tabelas do sistema bovespa para nova carga/atualização.
try:
    sql  = "SELECT COUNT(*) AS qt_rows FROM tb_capital;"
    rs = db.prepare(sql)
    for rl in rs:
        print ("Total de registros do ano " + str(dt_ano ) + " antes exclusao: " + str(rl[0]) + ";")
   
    #Monta sql para o deletar os registros
    sql  = "DELETE FROM tb_capital;"
    
    #Deleta todos os registros do ano referencia da tabela tb_capital
    db.execute(sql)
    #print(sql)
    print("Registros do ano " + str(dt_ano ) + " na tabela tb_capital foram excluidos;")

    sql  = "SELECT COUNT(*) AS qt_rows FROM tb_capital;"
    rs = db.prepare(sql)
    for rl in rs:
        print ("Total de registros do ano " + str(dt_ano ) + " apos exclusao: " + str(rl[0]) + ";")
    
except postgresql.exceptions.UniqueError: 
    print("Erro excluindo registros da tabela tb_capital;")

Total de registros do ano 2020 antes exclusao: 441;
Registros do ano 2020 na tabela tb_capital foram excluidos;
Total de registros do ano 2020 apos exclusao: 0;


In [5]:
def capital_sql(row):

    #Extrai e formata os campos do registro do arquivo txt para o sql  
    sq_capital      = "Nextval('sq_capital')"
    nm_pregao       = "'" + row[0].strip() + "'"
    cd_capital      = "'" + row[1].strip() + "'"
    dn_social       = "'" + row[2].strip() + "'"
    sg_mercado      = "'" + row[3].strip() + "'"
    tp_capital      = "'" + row[4].strip() + "'"
    vl_capital      = row[5].replace(".","").replace(",",".").strip()
    dt_aprovacao    = "'" + row[6][6:10] + "-" + row[6][3:5] + "-" + row[6][0:2] + "'"
    qt_ordinaria    = row[7].replace(".","").replace(",",".").strip()
    qt_preferencial = row[8].replace(".","").replace(",",".").strip()
    qt_total        = row[9].replace(".","").replace(",",".").strip()

    #Monta sql para o incluir o registro no banco de dados
    sql  = "Insert Into tb_capital ("
    sql += "cap_sq_capital, "          
    sql += "cap_nm_pregao, "       
    sql += "cap_cd_capital, "       
    sql += "cap_dn_social, "       
    sql += "cap_sg_mercado, "       
    sql += "cap_tp_capital, "       
    sql += "cap_vl_capital, "       
    sql += "cap_dt_aprovacao, "       
    sql += "cap_qt_ordinaria, "       
    sql += "cap_qt_preferencial, "     
    sql += "cap_qt_total "
    sql += ") Values ("
    sql += sq_capital + ", "
    sql += nm_pregao + ", "
    sql += cd_capital + ", "
    sql += dn_social + ", "
    sql += sg_mercado + ", "
    sql += tp_capital + ", "
    sql += vl_capital + ", "
    sql += dt_aprovacao + ", "
    sql += qt_ordinaria + ", "
    sql += qt_preferencial + ", "
    sql += qt_total + ");"

    #print(sql)
    return sql

In [6]:
#Insere os dados do arquivo cotacao historica na tabela tb_acao
with open("Data\CAPISOCI_A" + str(dt_ano) + ".TXT", "r") as fl:
    rd = csv.reader(fl, delimiter = "|")
    #Pula o registro de cabecario do arquivo
    next(rd)
    captype = "Outro"
    count = 0
    #le e inclui cada registro do arquivo csv na tabela
    for row in rd:
        #print(row[0].strip(), row)
        try:
            captype = row[4].strip()
        except ValueError:
            captype = "Outro"
        try:
            if(captype == "Homologado"):
                #monta comando SQL
                sql = capital_sql(row)
                #Inclue o registro do arquivo txt na tabela
                db.execute(sql)
                #print(sql)
                count += 1
        except postgresql.exceptions.UniqueError: 
            print("Erro inserindo registros da tabela tb_capital;")
#Fecha o arquivo csv
fl.close()
#Imprime total de registros inseridos
print("Total registros inseridos na tabela tb_capital: " + str(count) + ";")

Erro inserindo registros da tabela tb_capital;
Total registros inseridos na tabela tb_capital: 457;


In [7]:
#Imprime total de registros no banco de dados para a data referencia
try:
    sql  = "SELECT COUNT(*) AS qt_rows FROM tb_capital;"
    rs = db.prepare(sql)
    for rl in rs:
        print("Total de registros do ano " + str(dt_ano ) + " apos inclusao: " + str(rl[0]) + ";")
except postgresql.exceptions.UniqueError: 
    print("Erro pesquisando registros da tabela tb_capital;")

Total de registros do ano 2020 apos inclusao: 457;


In [8]:
#db.execute("COMMIT;")
print ("ok-commit")
db.close()
print ("ok-close")

ok-commit
ok-close


In [ ]:
#----------------------------------------------------------#
# That is all folks!                                       #
#----------------------------------------------------------#